# Baseline FastText — Classification Hiérarchique Rakuten

**Télécom Paris x Rakuten France** | Avril 2026

---

FastText est un modèle intégré (embedding sous-mots + classifieur linéaire) développé par Facebook Research.
Il est particulièrement adapté à la classification de texte e-commerce grâce à sa gestion des sous-mots
(robustesse aux fautes d'orthographe) et sa rapidité d'entraînement.

### Table des matières

1. [Configuration & Chargement](#section-1)
2. [Préparation des données FastText](#section-2)
3. [Classification L1](#section-3)
4. [Classification L2](#section-4)
5. [Métriques hiérarchiques](#section-5)
6. [Visualisations](#section-6)
7. [Conclusion](#section-7)


In [ ]:
# --- Configuration ---
import sys, time, warnings, tempfile, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score
from IPython.display import display

warnings.filterwarnings("ignore")
%matplotlib inline

sys.path.insert(0, str(Path.cwd().parent.parent))
from shared.data_loader import prepare_hierarchical_data_from_splits
from shared.metrics import (compute_all_metrics, compute_per_class_f1,
                             plot_confusion_matrix, plot_per_class_f1)
from shared.utils import set_seed

SEED = 42
N_JOBS = int(os.environ.get("N_JOBS", 26))  # Défaut 26, surchargeable via env
set_seed(SEED)

PROJECT_ROOT = Path.cwd().parent.parent
DATASET_DIR = Path("/home/infres/tandriam-25/fil_rouge/dataset")

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

# --- Fix compatibilité NumPy 2.x / fasttext ---
import fasttext
def _patched_predict(self, text, k=1, threshold=0.0, on_unicode_error="strict"):
    def check(entry):
        if entry.find("\n") != -1:
            raise ValueError("predict processes one line at a time (remove '\\n')")
        entry += "\n"
        return entry

    if type(text) == list:
        text = [check(entry) for entry in text]
        all_labels, all_probs = self.f.multilinePredict(text, k, threshold, on_unicode_error)
        return all_labels, all_probs
    else:
        text = check(text)
        predictions = self.f.predict(text, k, threshold, on_unicode_error)
        if predictions:
            probs, labels = zip(*predictions)
        else:
            probs, labels = ([], ())
        return labels, np.asarray(probs)

fasttext.FastText._FastText.predict = _patched_predict

print(f"Configuration terminée. N_JOBS={N_JOBS}")

In [ ]:
# --- Chargement des données ---
print("Chargement des splits train/val/test...")
t0 = time.time()

data = prepare_hierarchical_data_from_splits(
    train_path=str(DATASET_DIR / "train_split.jsonl"),
    val_path=str(DATASET_DIR / "val_split.jsonl"),
    test_path=str(DATASET_DIR / "test_split.jsonl"),
    n_samples=None,
    seed=SEED,
)

print(f"Chargement terminé en {time.time() - t0:.0f}s")
print(f"  Train : {len(data['X_train']):,} | Val : {len(data['X_val']):,} | Test : {len(data['X_test']):,}")
print(f"  L1 : {data['num_classes_l1']} | L2 : {data['num_classes_l2']}")


<a id="section-2"></a>
## 2. Préparation des données FastText

FastText attend un fichier texte avec le format : `__label__<classe> <texte>`

Chaque ligne contient un label préfixé par `__label__` suivi du texte du produit.


In [ ]:
# --- Conversion au format FastText ---
def write_fasttext_file(texts, labels, encoder, filepath):
    """Écrit les données au format FastText."""
    label_names = encoder.inverse_transform(labels)
    with open(filepath, "w", encoding="utf-8") as f:
        for text, label in zip(texts, label_names):
            clean_label = str(label).replace(" ", "_").replace("\n", "_")
            clean_text = str(text).replace("\n", " ").strip()
            f.write(f"__label__{clean_label} {clean_text}\n")

tmpdir = tempfile.mkdtemp()

# Fichiers L1
train_l1_path = os.path.join(tmpdir, "train_l1.txt")
test_l1_path = os.path.join(tmpdir, "test_l1.txt")
write_fasttext_file(data["X_train"], data["y_l1_train"], data["l1_encoder"], train_l1_path)
write_fasttext_file(data["X_test"], data["y_l1_test"], data["l1_encoder"], test_l1_path)

# Fichiers L2
train_l2_path = os.path.join(tmpdir, "train_l2.txt")
test_l2_path = os.path.join(tmpdir, "test_l2.txt")
write_fasttext_file(data["X_train"], data["y_l2_train"], data["l2_encoder"], train_l2_path)
write_fasttext_file(data["X_test"], data["y_l2_test"], data["l2_encoder"], test_l2_path)

# Vérification
for name, path in [("train_l1", train_l1_path), ("test_l1", test_l1_path),
                    ("train_l2", train_l2_path), ("test_l2", test_l2_path)]:
    n_lines = sum(1 for _ in open(path))
    print(f"  {name}: {n_lines:,} lignes")


<a id="section-3"></a>
## 3. Classification L1 (157 catégories)

Entraînement avec `loss="softmax"` (adapté pour le multiclass standard),
sous-mots de caractères et bigrammes de mots (`wordNgrams=2`).


In [ ]:
# --- Entraînement FastText L1 ---
print(f"Entraînement FastText L1 ({N_JOBS} threads)...")
t0 = time.time()

model_l1 = fasttext.train_supervised(
    input=train_l1_path,
    lr=0.5,
    epoch=25,
    wordNgrams=2,
    dim=100,
    loss="softmax",
    thread=N_JOBS,
    verbose=0,
)
train_time_l1 = time.time() - t0
print(f"Entraînement L1 terminé en {train_time_l1:.0f}s")

# --- Prédiction batch ---
def fasttext_predict_batch(model, texts, encoder, label=""):
    """Prédit avec FastText en batch et convertit en indices encodés."""
    class_to_idx = {c: i for i, c in enumerate(encoder.classes_)}
    cleaned = [str(t).replace("\n", " ").strip() for t in texts]

    print(f"  {label} prédiction batch sur {len(cleaned):,} textes...")
    labels_batch, _ = model.predict(cleaned)

    preds = []
    for labels in labels_batch:
        name = labels[0].replace("__label__", "").replace("_", " ")
        preds.append(class_to_idx.get(name, 0))
    return np.array(preds)

print("Prédiction L1 sur le test set...")
preds_l1_test = fasttext_predict_batch(model_l1, data["X_test"], data["l1_encoder"], label="L1")
f1_l1 = f1_score(data["y_l1_test"], preds_l1_test, average="macro")
print(f"FastText L1 — Test F1 macro : {f1_l1:.4f}")

<a id="section-4"></a>
## 4. Classification L2 (6000+ sous-catégories)

Pour le L2 avec 6000+ classes, on utilise `loss="ova"` (one-vs-all) qui est plus
adapté aux problèmes avec un très grand nombre de classes.


In [ ]:
# --- Entraînement FastText L2 ---
# Réduit à 10 epochs (vs 25 pour L1) : OVA crée 6000+ classifieurs binaires,
# 25 epochs × 5.5M × 6000+ → mémoire excessive
# thread limité à 10 pour L2 OVA : trop de threads cause des NaN
# (race conditions dans l'accumulation des gradients de 6000+ classifieurs)
L2_THREADS = min(N_JOBS, 10)
print(f"Entraînement FastText L2 (6000+ classes, {L2_THREADS} threads pour OVA)...")
t0 = time.time()

model_l2 = fasttext.train_supervised(
    input=train_l2_path,
    lr=0.5,
    epoch=10,
    wordNgrams=2,
    dim=100,
    loss="ova",
    thread=L2_THREADS,
    verbose=0,
)
train_time_l2 = time.time() - t0
print(f"Entraînement L2 terminé en {train_time_l2:.0f}s")

print("Prédiction L2 sur le test set...")
preds_l2_test = fasttext_predict_batch(model_l2, data["X_test"], data["l2_encoder"], label="L2")
f1_l2 = f1_score(data["y_l2_test"], preds_l2_test, average="macro")
print(f"FastText L2 — Test F1 macro : {f1_l2:.4f}")

<a id="section-5"></a>
## 5. Métriques hiérarchiques

FastText entraîne L1 et L2 indépendamment — il n'y a aucune contrainte hiérarchique.


In [ ]:
# --- Métriques hiérarchiques ---
all_metrics = compute_all_metrics(
    labels_l1=data["y_l1_test"], preds_l1=preds_l1_test,
    labels_l2=data["y_l2_test"], preds_l2=preds_l2_test,
    l2_to_l1_mapping=data["l2_to_l1_mapping"],
)

print("=== Métriques FastText (Test Set) ===")
for k, v in all_metrics.items():
    if isinstance(v, float):
        print(f"  {k:30s} : {v:.4f}")

# Tableau récapitulatif
summary = pd.DataFrame([{
    "Modèle": "FastText",
    "F1 L1 macro": all_metrics.get("f1_l1_macro", f1_l1),
    "F1 L2 macro": all_metrics.get("f1_l2_macro", f1_l2),
    "HCR": all_metrics.get("hcr", 0),
    "Temps total": f"{train_time_l1 + train_time_l2:.0f}s",
}])
display(summary.style.format({
    "F1 L1 macro": "{:.4f}", "F1 L2 macro": "{:.4f}", "HCR": "{:.4f}",
}).set_caption("Résultats FastText (Test Set)"))


<a id="section-6"></a>
## 6. Visualisations


In [ ]:
# --- Matrice de confusion L1 + F1 par classe ---
per_class_f1_l1 = compute_per_class_f1(data["y_l1_test"], preds_l1_test, data["l1_encoder"])

fig_cm = plot_confusion_matrix(
    data["y_l1_test"], preds_l1_test, data["l1_encoder"],
    title="Matrice de confusion L1 — FastText", top_n=30,
)
plt.show()

fig_f1 = plot_per_class_f1(per_class_f1_l1, title="F1 par classe L1 — FastText", top_n=20)
plt.show()


In [ ]:
# --- F1 par classe L2 ---
per_class_f1_l2 = compute_per_class_f1(data["y_l2_test"], preds_l2_test, data["l2_encoder"])

fig_f1_l2 = plot_per_class_f1(per_class_f1_l2, title="F1 par classe L2 — FastText", top_n=20)
plt.show()

f1_vals = list(per_class_f1_l2.values())
print(f"F1 L2 — Moyenne: {np.mean(f1_vals):.4f}, Médiane: {np.median(f1_vals):.4f}")
print(f"  Classes avec F1 = 0 : {sum(1 for v in f1_vals if v == 0)}")
print(f"  Classes avec F1 > 0.5 : {sum(1 for v in f1_vals if v > 0.5)}")


<a id="section-7"></a>
## 7. Conclusion

### Forces de FastText
- **Ultra-rapide** : entraînement en quelques minutes même sur 5.5M de textes
- **Sous-mots** : capture les morphèmes (préfixes, suffixes) et résiste aux fautes d'orthographe
- **Modèle intégré** : embedding + classifieur entraînés conjointement (end-to-end)

### Limites
- **Pas contextuel** : un mot a toujours le même vecteur quel que soit le contexte
- **Pas hiérarchique** : L1 et L2 sont entraînés indépendamment
- **Classifieur linéaire** : ne capture pas les interactions complexes entre features

---
*Notebook généré dans le cadre du projet de classification hiérarchique Rakuten — Télécom Paris x Rakuten France, 2026.*


In [ ]:
# --- Nettoyage des fichiers temporaires ---
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
print("Fichiers temporaires supprimés.")
